# 06 - Baseline Models

This notebook trains baseline models for the two project tasks: clinical utility and linkability.


## Pipeline Overview

- Load ECG-derived segment features
- Prepare train/test splits with no patient leakage
- Train utility baselines: Logistic Regression and XGBoost when available
- Generate balanced positive/negative segment pairs for linkability
- Train linkability baselines using pairwise representations
- Summarize baseline performance


## Selected Segmentation

This notebook uses the selected segmentation configuration from `05_segmentation_selection.ipynb`:
- `window_sec = 2.0`
- `overlap = 0.5`
- `step_sec = 1.0`

The canonical feature dataset is loaded from `data/processed/final_segment_features_dataset`.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

import importlib
import pandas as pd
import modeling

importlib.reload(modeling)

from config import FINAL_SEGMENT_FEATURES_DIR
from modeling import run_linkability_baselines, run_utility_baselines, run_utility_baselines_tuned


In [2]:
FEATURES_DATASET_DIR = FINAL_SEGMENT_FEATURES_DIR
MAX_CHUNKS = None  # set an integer here for quick tests
UTILITY_MODELS = ["LogisticRegression", "XGBoost"]
UTILITY_TUNED_MODELS = ["LogisticRegression", "XGBoost"]


## Load Data


In [3]:
manifest_path = FEATURES_DATASET_DIR / "manifest.json"
errors_path = FEATURES_DATASET_DIR / "errors.csv"

manifest = pd.read_json(manifest_path, typ="series")
chunk_files = [FEATURES_DATASET_DIR / chunk["chunk_file"] for chunk in manifest["chunks"]]
if MAX_CHUNKS is not None:
    chunk_files = chunk_files[:MAX_CHUNKS]

features_df = pd.concat(
    [pd.read_csv(chunk_file, compression="gzip") for chunk_file in chunk_files],
    ignore_index=True,
)
errors_df = pd.read_csv(errors_path) if errors_path.exists() and errors_path.stat().st_size > 0 else pd.DataFrame()

print("Chunk files loaded:", len(chunk_files))
print("Features dataframe:", features_df.shape)
print("Stored preprocessing errors:", len(errors_df))
features_df.head()


Chunk files loaded: 181
Features dataframe: (406359, 215)
Stored preprocessing errors: 1


,patient_id,segment_id,label,utility_label,segment_ref,start_sample,end_sample,lead_I_mean,lead_I_std,lead_I_min,...,global_min_rms,global_max_rms,global_mean_zero_crossing_rate,global_std_zero_crossing_rate,global_min_zero_crossing_rate,global_max_zero_crossing_rate,global_mean_n_peaks,global_std_n_peaks,global_min_n_peaks,global_max_n_peaks
0,JS00001,JS00001_seg_0000,"164889003,59118001,164934002",Atrial,0,0,1000,-0.163769,1.209824,-4.846666,...,0.866558,1.490585,0.023357,0.007517,0.014014,0.038038,6.083333,1.114924,5.0,8.0
1,JS00001,JS00001_seg_0001,"164889003,59118001,164934002",Atrial,1,500,1500,0.004842,0.836291,-2.321608,...,0.793182,1.265454,0.024358,0.009794,0.011011,0.039039,5.416667,1.381927,3.0,7.0
2,JS00001,JS00001_seg_0002,"164889003,59118001,164934002",Atrial,2,1000,2000,0.022468,0.860830,-2.385116,...,0.786630,1.548732,0.028195,0.013632,0.009009,0.046046,5.916667,1.320248,3.0,7.0
3,JS00001,JS00001_seg_0003,"164889003,59118001,164934002",Atrial,3,1500,2500,0.115216,0.908847,-2.936894,...,0.859385,1.348393,0.031031,0.014791,0.009009,0.049049,6.666667,1.312335,4.0,8.0
4,JS00001,JS00001_seg_0004,"164889003,59118001,164934002",Atrial,4,2000,3000,0.046629,0.995601,-2.936894,...,0.787450,1.122386,0.027861,0.010505,0.015015,0.051051,6.166667,1.280191,4.0,8.0


## Baseline de Utilidade


In [4]:
utility_results_base = run_utility_baselines(
    features_df=features_df,
    target_col="utility_label",
    group_col="patient_id",
    test_size=0.2,
    random_state=42,
    models=UTILITY_MODELS,
)

utility_results_base["summary_df"]


,model,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc
0,LogisticRegression,0.67890,0.780105,0.831770,0.907671,0.712424
1,XGBoost,0.67526,0.798605,0.775427,0.925902,0.779992


In [5]:
for result in utility_results_base["results"]:
    print(f"\n=== {result['model']} ===")
    print("Confusion matrix:")
    print(result["confusion_matrix"])
    print("Classification report:")
    print(result["classification_report"])


=== LogisticRegression ===
Confusion matrix:
[[52300 11267]
 [ 2820 14892]]
Classification report:
              precision    recall  f1-score   support

  Non-Atrial       0.95      0.82      0.88     63567
      Atrial       0.57      0.84      0.68     17712

    accuracy                           0.83     81279
   macro avg       0.76      0.83      0.78     81279
weighted avg       0.87      0.83      0.84     81279


=== XGBoost ===
Confusion matrix:
[[60415  3152]
 [ 7077 10635]]
Classification report:
              precision    recall  f1-score   support

  Non-Atrial       0.90      0.95      0.92     63567
      Atrial       0.77      0.60      0.68     17712

    accuracy                           0.87     81279
   macro avg       0.83      0.78      0.80     81279
weighted avg       0.87      0.87      0.87     81279



## Baseline de Utilidade Afinada


In [ ]:
utility_results_tuned = run_utility_baselines_tuned(
    features_df=features_df,
    target_col="utility_label",
    group_col="patient_id",
    test_size=0.2,
    val_size=0.2,
    random_state=42,
    threshold_grid=[0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6],
    xgb_params={
        "max_depth": 4,
        "learning_rate": 0.1,
        "n_estimators": 100,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    models=UTILITY_TUNED_MODELS,
)

utility_results_tuned["summary_df"]


In [ ]:
for result in utility_results_tuned["results"]:
    print(f"\n=== {result['model']} ===")
    print(f"Threshold: {result['threshold']:.2f}")
    print("Confusion matrix:")
    print(result["confusion_matrix"])
    print("Classification report:")
    print(result["classification_report"])


## Baseline de Linkability


In [6]:
linkability_results = run_linkability_baselines(
    features_df=features_df,
    group_col="patient_id",
    test_size=0.2,
    random_state=42,
    max_pairs=2000,
    representation="absdiff",
    min_segment_gap=4,
)

linkability_results["summary_df"]


,model,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc
0,LogisticRegression,0.975219,0.975250,0.97525,0.996190,0.996639
1,XGBoost,0.978809,0.978998,0.97900,0.998343,0.998426


In [7]:
print("Train pairs:", linkability_results["train_pair_df"].shape)
print("Test pairs:", linkability_results["test_pair_df"].shape)
print("Pair feature matrix train:", linkability_results["X_train"].shape)
print("Pair feature matrix test:", linkability_results["X_test"].shape)


Train pairs: (4000, 3)
Test pairs: (4000, 3)
Pair feature matrix train: (4000, 208)
Pair feature matrix test: (4000, 208)


In [8]:
for result in linkability_results["results"]:
    print(f"\n=== {result['model']} ===")
    print("Confusion matrix:")
    print(result["confusion_matrix"])
    print("Classification report:")
    print(result["classification_report"])



=== LogisticRegression ===
Confusion matrix:
[[1953   47]
 [  52 1948]]
Classification report:
                   precision    recall  f1-score   support

different_patient       0.97      0.98      0.98      2000
     same_patient       0.98      0.97      0.98      2000

         accuracy                           0.98      4000
        macro avg       0.98      0.98      0.98      4000
     weighted avg       0.98      0.98      0.98      4000


=== XGBoost ===
Confusion matrix:
[[1976   24]
 [  60 1940]]
Classification report:
                   precision    recall  f1-score   support

different_patient       0.97      0.99      0.98      2000
     same_patient       0.99      0.97      0.98      2000

         accuracy                           0.98      4000
        macro avg       0.98      0.98      0.98      4000
     weighted avg       0.98      0.98      0.98      4000



## Baseline Decision

Primary baselines for the next stage of the project:

- Utility: `LogisticRegression`
- Linkability: `XGBoost`

Rationale:

- `LogisticRegression` is the preferred utility baseline because it stays highly competitive while being simpler and more interpretable.
- `XGBoost` is the preferred linkability baseline because it gives the strongest discrimination for same-patient vs different-patient pairs.
- The other baseline models remain useful as secondary comparisons, but they are not the main reference models for the privacy-utility analysis.


## Final Results


In [9]:
utility_summary_base = utility_results_base["summary_df"].copy()
utility_summary_base["task"] = "utility_base"

linkability_summary = linkability_results["summary_df"].copy()
linkability_summary["task"] = "linkability"

selected_baselines_df = pd.concat([
    utility_summary_base[utility_summary_base["model"] == "LogisticRegression"],
    linkability_summary[linkability_summary["model"] == "XGBoost"],
], ignore_index=True)

all_baselines_df = pd.concat([utility_summary_base, linkability_summary], ignore_index=True)

print("Selected baselines for next stage:")
selected_baselines_df


Selected baselines for next stage:


,model,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc,task
0,LogisticRegression,0.678900,0.780105,0.83177,0.907671,0.712424,utility_base
1,XGBoost,0.978809,0.978998,0.97900,0.998343,0.998426,linkability
